# IZQAR PaySim — validation-only recall candidate

Attach Kaggle input `ealaxi/paysim1`. This notebook evaluates exactly one candidate: an XGBoost PaySim model with `positive_weight_multiplier=1.25`. It uses chronological train/validation/test boundaries but **does not predict, report, or select anything on the final test window**. The archive contains only a report, never an installable serving model.

Before using it, push the required commit to the feature branch or upload equivalent current source. The revision assertion intentionally prevents a silent run of stale GitHub code. Compare this candidate only with a same-split experiment baseline, using validation Recall, PR-AUC, false alerts, `TRANSFER`/`CASH_OUT` slices, and preprocessing stress diagnostics.

In [ ]:
from pathlib import Path
import subprocess, sys

EXPECTED_REVISION = '5111638e3e3a4ae1c8aee948525de0a95fd5d679'
root = Path('/kaggle/working/izqar_paysim_candidate')
if root.exists():
    raise FileExistsError(f'{root} already exists; use a fresh Kaggle session to avoid stale code')
subprocess.run(['git', 'clone', '--branch', 'feat/ml-pipelines-and-startup',
                'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
actual_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
if actual_revision != EXPECTED_REVISION:
    raise RuntimeError(f'Feature branch is {actual_revision}, expected {EXPECTED_REVISION}. Push the reviewed feature commit first; do not train stale code.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r',
                str(root / 'backend' / 'requirements-ml.txt')], check=True)
matches = list(Path('/kaggle/input').rglob('PS_20174392719_1491204439457_log.csv'))
if len(matches) != 1:
    raise FileNotFoundError(f'Expected one PaySim CSV under /kaggle/input, found {len(matches)}. Add `ealaxi/paysim1`.')
print('PaySim input:', matches[0], 'bytes:', matches[0].stat().st_size, 'code:', actual_revision)


In [ ]:
import json, zipfile
backend = root / 'backend'
output = Path('/kaggle/working/paysim_recall_weight125_candidate.joblib')
command = [sys.executable, str(backend / 'scripts' / 'train_paysim.py'),
           '--data', str(matches[0]), '--output', str(output), '--estimators', '180',
           '--full-data', '--mode', 'experiment', '--model-version', 'paysim-xgboost-v3',
           '--positive-weight-multiplier', '1.25']
subprocess.run(command, cwd=backend, check=True)
report = output.with_suffix('.metrics.json')
metadata = json.loads(report.read_text(encoding='utf-8'))
assert not output.exists(), 'Experiment mode must not write a serving model'
assert metadata['run_mode'] == 'experiment'
assert metadata['test_metrics'] is None and metadata['baseline_test_metrics'] is None
assert metadata['experiment']['hyperparameters']['positive_weight_multiplier'] == 1.25
assert set(metadata['validation_type_slices']) == {'TRANSFER', 'CASH_OUT'}
assert metadata['validation_robustness']['scope'] == 'validation_only_synthetic_preprocessing_stress'
print(json.dumps({
    'validation_metrics': metadata['validation_metrics'],
    'type_slices': metadata['validation_type_slices'],
    'operating_profiles': metadata['validation_operating_profiles'],
    'experiment_id': metadata['experiment']['experiment_id'],
}, ensure_ascii=False, indent=2))


In [ ]:
bundle = Path('/kaggle/working/izqar_paysim_recall_candidate_report.zip')
if bundle.exists():
    raise FileExistsError(f'Refusing to overwrite {bundle}')
run_manifest = Path('/kaggle/working/paysim_recall_candidate_run.json')
run_manifest.write_text(json.dumps({
    'expected_revision': EXPECTED_REVISION, 'actual_revision': actual_revision,
    'dataset_input_file': matches[0].name, 'dataset_input_bytes': matches[0].stat().st_size,
    'mode': 'experiment', 'candidate': 'paysim_positive_weight_1.25',
    'test_evaluation_performed': False,
}, ensure_ascii=False, indent=2), encoding='utf-8')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(report, report.name)
    archive.write(run_manifest, run_manifest.name)
print('Download report-only candidate ZIP from Kaggle Output:', bundle)
